In [ ]:
import threading
import time

import numpy as np
import anywidget_instruments as ai

probe = ai.Gauge(0, label="Perf probe", max=1000)
knob = ai.Knob(50, step=1, label="Perf knob")
gauges = [ai.Gauge(0, label=f"g{i}", size=(70, 70)) for i in range(50)]
chart = ai.WaveformChart(label="Perf chart", history=2000, autoscale_y=True, size=(420, 200))
display(ai.Panel([probe, knob, chart], columns=3))
display(ai.Panel(gauges, columns=10))
stamps = {}


def probe_run(n=100, period=0.05):
    """PERF-001: timestamps of kernel-side updates."""
    def run():
        for i in range(1, n + 1):
            stamps[i] = time.time() * 1000
            probe.value = i
            time.sleep(period)
    threading.Thread(target=run, daemon=True).start()


indicators_stop = threading.Event()


def indicators_run(seconds=4.0):
    """PERF-002: 50 indicators updated at 20 Hz (until ``seconds`` or indicators_stop)."""
    indicators_stop.clear()
    def run():
        t0 = time.time()
        k = 0
        while time.time() - t0 < seconds and not indicators_stop.is_set():
            for i, g in enumerate(gauges):
                g.value = 50 + 40 * np.sin(k / 5 + i)
            k += 1
            time.sleep(0.05)
    threading.Thread(target=run, daemon=True).start()


def chart_run(seconds=4.0):
    """CHART-005: samples appended at 1 kHz (blocks of 50 every 50 ms)."""
    def run():
        t0 = time.time()
        k = 0
        while time.time() - t0 < seconds:
            chart.append(np.sin(np.arange(k * 50, (k + 1) * 50) / 40))
            k += 1
            time.sleep(0.05)
    threading.Thread(target=run, daemon=True).start()